# Chapter 6: Maneuvering Models

## Section 6.1 & 6.2: Rigid-Body Kinetics ($M_{RB}$)

Maneuvering models focus on low-frequency, large-amplitude displacements in calm water. In Section 6.2, Fossen derives the dry structural mass properties of the marine craft. 

If we align our body-fixed coordinate origin with the submarine's Center of Buoyancy ($CB$), and the Center of Gravity ($G$) sits directly below it at coordinates $r_g = [0, 0, z_g]^T$, the 6-DOF rigid-body mass matrix $M_{RB}$ is structured as:

$$M_{RB} = \begin{bmatrix} 
m & 0 & 0 & 0 & mz_g & 0 \\ 
0 & m & 0 & -mz_g & 0 & 0 \\ 
0 & 0 & m & 0 & 0 & 0 \\ 
0 & -mz_g & 0 & I_x & 0 & 0 \\ 
mz_g & 0 & 0 & 0 & I_y & 0 \\ 
0 & 0 & 0 & 0 & 0 & I_z 
\end{bmatrix}$$

### What We Are Experimenting With:
By adjusting the vertical ballast location ($z_g$), you are changing how low the dry weight sits beneath the center of volume. Notice how changing this single value immediately updates the cross-coupling indices between linear inertia and rotational inertia.

In [1]:
import numpy as np
import ipywidgets as widgets

def inspect_section_6_2_mass(z_g_val):
    """
    Builds and displays the pure rigid-body mass matrix M_RB to isolate
    the inertial coupling terms caused by vertical center of gravity offsets.
    """
    m = 1500000.0  # 1500 metric tons
    ix, iy, iz = 2.0e6, 4.5e7, 4.5e7
    
    M_RB = np.array([
        [m,   0.0, 0.0, 0.0,   m*z_g_val, 0.0],
        [0.0, m,   0.0, -m*z_g_val, 0.0,   0.0],
        [0.0, 0.0, m,   0.0,   0.0,   0.0],
        [0.0, -m*z_g_val, 0.0, ix,  0.0,   0.0],
        [m*z_g_val,  0.0, 0.0, 0.0,  iy,    0.0],
        [0.0, 0.0, 0.0, 0.0,   0.0,   iz]
    ])
    
    print("=======================================================================")
    print(f"      RIGID BODY MASS MATRIX M_RB (Fossen Section 6.2) for z_g = {z_g_val} m")
    print("=======================================================================")
    with np.printoptions(precision=1, suppress=True):
        print(M_RB)
    print("-----------------------------------------------------------------------")
    print(f"PATTERN TO NOTICE:")
    print(f"-> At index [0,4] and [4,0], Surge and Pitch are coupled by: {m*z_g_val:,.1f} kg·m")
    print(f"-> At index [1,3] and [3,1], Sway and Roll are coupled by: {-m*z_g_val:,.1f} kg·m")
    print("If z_g = 0 (CG perfectly matches CB), these cross-coupling terms completely vanish.")

widgets.interact(inspect_section_6_2_mass,
                 z_g_val=widgets.FloatSlider(min=0.0, max=1.5, step=0.1, value=0.4, description='CG Depth z_g:'));

interactive(children=(FloatSlider(value=0.4, description='CG Depth z_g:', max=1.5), Output()), _dom_classes=('…

---

## Section 6.3: Potential-Fluid Hydrodynamics & The Munk Moment ($C_A(\nu)$)

When moving through water, an elongated hull experiences a non-dissipative fluid acceleration penalty called **Added Mass ($A$)**. Because a submarine is a slender cylinder, broadside added mass terms like $Y_{\dot{v}}$ are massive, while forward surge added mass $X_{\dot{u}}$ is tiny.

### The Munk Moment
When the submarine moves forward with a small angle of drift (slipping slightly sideways), this imbalance generates a highly destabilizing fluid torque in the Coriolis matrix $C_A(\nu)$. This is the **Munk Moment**, which acts to twist the submarine broadside-first into the incoming fluid flow.



### What We Are Experimenting With:
Use the slider to accelerate the submarine's forward cruise speed ($u$). Notice the mathematical pattern: the destabilizing turning torque scales **quadratically** with speed. Double your forward speed, and the unforced twist ripping your bow off course quadruples!

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

def plot_section_6_3_munk_moment(forward_speed_u):
    """
    Calculates and plots the destabilizing Munk Moment torque across a 
    spectrum of lateral drift velocities (v) to evaluate hydrodynamic instability.
    """
    m = 1500000.0
    X_udot = m * 0.1  # 10% added mass longitudinally
    Y_vdot = m * 0.8  # 80% added mass broadside due to massive water displacement
    
    drift_velocities = np.linspace(-1.5, 1.5, 100) # Sway velocity range (m/s)
    munk_torques = []
    
    for v in drift_velocities:
        # Fossen Section 6.3 torque derivation: Torque = (X_udot - Y_vdot) * u * v
        torque = (X_udot - Y_vdot) * forward_speed_u * v
        munk_torques.append(torque)
        
    plt.figure(figsize=(9, 4))
    plt.plot(drift_velocities, np.array(munk_torques) / 1e3, color='crimson', linewidth=2.5, label='Munk Moment ($C_A$)')
    plt.axhline(0, color='black', linestyle=':', alpha=0.5)
    plt.axvline(0, color='black', linestyle=':', alpha=0.5)
    
    plt.title(f"Section 6.3 Destabilizing Munk Moment @ Forward Speed u = {forward_speed_u} m/s", fontsize=11)
    plt.xlabel("Lateral Drift Velocity $v$ (Sway Slippage) [m/s]")
    plt.ylabel("Unforced Destabilizing Torque [kN·m]")
    plt.grid(True, linestyle=':', alpha=0.6)
    
    # Calculate a specific callout value for clarity
    sample_v = 0.5
    sample_torque = (X_udot - Y_vdot) * forward_speed_u * sample_v / 1e3
    plt.scatter(sample_v, sample_torque, color='black', zorder=5)
    plt.annotate(f"At 0.5 m/s sway slip:\n{sample_torque:,.0f} kN·m twisting bow", 
                 xy=(sample_v, sample_torque), xytext=(sample_v + 0.1, sample_torque - 1000),
                 arrowprops=dict(arrowstyle="->", color='black'))
    
    plt.show()

widgets.interact(plot_section_6_3_munk_moment,
                 forward_speed_u=widgets.FloatSlider(min=0.0, max=8.0, step=0.5, value=4.0, description='Forward u [m/s]:'));

interactive(children=(FloatSlider(value=4.0, description='Forward u [m/s]:', max=8.0, step=0.5), Output()), _d…

---

## Section 6.4: Dissipative Hydrodynamic Forces (Damping: $D(\nu)$)

To stop the destabilizing Munk Moment from instantly spinning the vehicle out of control, we introduce viscous fluid damping. Fossen formalizes this by splitting the matrix into a linear and non-linear component:

$$D(\nu) = D_l + D_n(\nu)$$

1. **Linear Damping ($D_l$):** Captures low-speed skin friction, dominating during fine docking maneuvers or dead-slow hovering.
2. **Non-linear Damping ($D_n(\nu)$):** Models quadratic cross-flow drag and vortex shedding at transit speeds, scaling directly with the absolute value of the vehicle's speed.

### What We Are Experimenting With:
By scaling the `Crossflow Drag Coefficient (D_n)`, you can visualize the exact angular velocity transition where linear damping stops mattering and quadratic cross-flow fluid drag completely takes over the hull profile.

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

def plot_section_6_4_damping_regimes(D_n_yaw_multiplier):
    """
    Visualizes the transition from linear skin friction to quadratic cross-flow drag 
    in Yaw Damping as a function of the vessel's rotational velocity.
    """
    yaw_rates = np.linspace(0, 0.20, 100) # rad/s tracking speed
    
    D_l_yaw = 2.0e5  # Base linear coefficient
    D_n_yaw = 8.0e5 * D_n_yaw_multiplier  # Base quadratic crossflow coefficient
    
    total_drag_force = []
    linear_component = []
    
    for r in yaw_rates:
        lin = D_l_yaw * r
        nonlin = D_n_yaw * (r**2)
        linear_component.append(lin)
        total_drag_force.append(lin + nonlin)
        
    plt.figure(figsize=(9, 4))
    plt.plot(np.degrees(yaw_rates), np.array(total_drag_force) / 1e3, color='darkviolet', linewidth=2.5, label='Total Viscous Damping $D(\\nu)\\nu$')
    plt.plot(np.degrees(yaw_rates), np.array(linear_component) / 1e3, 'k--', alpha=0.5, label='Linear Friction Component ($D_l$)')
    
    plt.title("Section 6.4 Fluid Damping: Linear vs. Quadratic Blending", fontsize=11)
    plt.xlabel("Rotational Turning Rate $r$ [deg/s]")
    plt.ylabel("Resisting Damping Torque [kN·m]")
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend(loc='upper left')
    plt.show()

widgets.interact(plot_section_6_4_damping_regimes,
                 D_n_yaw_multiplier=widgets.FloatSlider(min=0.2, max=3.0, step=0.2, value=1.0, description='Crossflow D_n %:'));

interactive(children=(FloatSlider(value=1.0, description='Crossflow D_n %:', max=3.0, min=0.2, step=0.2), Outp…

---

## Section 6.5 & 6.6: Unified System Representations

Section 6.5 maps the gravity and buoyancy forces into a vector $g(\eta)$. For a fully submerged submarine maintaining neutral buoyancy, net vertical forces cancel out, leaving only the righting moments in Roll and Pitch driven by the vertical distance ($z_g$).

Section 6.6 unifies every single preceding section into Fossen's standard monolithic matrix formulation:

$$(M_{RB} + A)\dot{\nu} + \big(C_{RB}(\nu) + C_A(\nu)\big)\nu + \big(D_l + D_n(\nu)\big)\nu + g(\eta) = \tau$$

### What We Are Experimenting With:
Let's see how these pieces combine over time. By passing an initial angular turn error into this monolithic equation, you can test how changing your **Added Mass Percentage** changes the inertia of the system. 

Look for the pattern: Higher added mass percentages act as an extra heavy weight. It slows down the rate of your acceleration, causing the boat to feel much more sluggish when reacting to disturbances.

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

def simulate_section_6_6_unforced_system(added_mass_multiplier, init_yaw_kick_deg):
    """
    Executes an explicit time-domain simulation of Fossen's Section 6.6 unified form 
    across 6-DOF to observe how hydrodynamics damp out initial angular errors.
    """
    t = np.linspace(0, 30, 200)
    dt = t[1] - t[0]
    
    m = 1500000.0
    ix, iy, iz = 2.0e6, 4.5e7, 4.5e7
    z_g = 0.4
    W = m * 9.81
    
    # Section 6.2 Matrix
    M_RB = np.array([
        [m,   0.0, 0.0, 0.0,   m*z_g, 0.0],
        [0.0, m,   0.0, -m*z_g, 0.0,   0.0],
        [0.0, 0.0, m,   0.0,   0.0,   0.0],
        [0.0, -m*z_g, 0.0, ix,  0.0,   0.0],
        [m*z_g,  0.0, 0.0, 0.0,  iy,    0.0],
        [0.0, 0.0, 0.0, 0.0,   0.0,   iz]
    ])
    
    # Section 6.3 Added Mass terms scaled by slider
    X_udot = m * 0.1
    Y_vdot = m * 0.8 * added_mass_multiplier
    W_wdot = m * 0.8 * added_mass_multiplier
    K_pdot = ix * 0.2
    M_qdot = iy * 0.7 * added_mass_multiplier
    N_rdot = iz * 0.7 * added_mass_multiplier
    
    A = np.diag([X_udot, Y_vdot, W_wdot, K_pdot, M_qdot, N_rdot])
    M_total = M_RB + A
    M_inv = np.linalg.inv(M_total)
    
    # State vectors
    nu = np.zeros((6, len(t)))
    # Set initial condition: 3.5 m/s forward speed with user-defined rotational error kick
    nu[:, 0] = [3.5, 0.0, 0.0, 0.0, 0.0, np.radians(init_yaw_kick_deg)] 
    
    pitch = np.zeros_like(t)
    roll = np.zeros_like(t)
    
    D_l = np.diag([5e3, 3e4, 3e4, 1e4, 2e5, 2e5])
    D_n_base = np.diag([2e3, 5e4, 5e4, 2e3, 8e5, 8e5])
    
    for i in range(1, len(t)):
        u, v, w, p, q, r = nu[:, i-1]
        
        # Section 6.3 Coriolis Cross-coupling update
        a1, a2, a3 = X_udot*u, Y_vdot*v, W_wdot*w
        a4, a5, a6 = K_pdot*p, M_qdot*q, N_rdot*r
        C_A = np.array([
            [0.0,   0.0,   0.0,   0.0,   a3,   -a2],
            [0.0,   0.0,   0.0,   -a3,   0.0,   a1],
            [0.0,   0.0,   0.0,   a2,    -a1,   0.0],
            [0.0,   a3,    -a2,   0.0,   a6,   -a5],
            [-a3,   0.0,   a1,    -a6,   0.0,   a4],
            [a2,    -a1,   0.0,   a5,    -a4,   0.0]
        ])
        
        D_total = D_l + D_n_base * np.linalg.norm(nu[:, i-1])
        g_vector = np.array([0.0, 0.0, 0.0, z_g*W*np.cos(pitch[i-1])*np.sin(roll[i-1]), z_g*W*np.sin(pitch[i-1]), 0.0])
        
        # Fossen Section 6.6 Acceleration Solve Step
        nu_dot = M_inv @ (-C_A @ nu[:, i-1] - D_total @ nu[:, i-1] - g_vector)
        nu[:, i] = nu[:, i-1] + nu_dot * dt
        
        roll[i] = roll[i-1] + p * dt
        pitch[i] = pitch[i-1] + q * dt

    # Plot velocity tracking responses
    plt.figure(figsize=(10, 4.5))
    plt.plot(t, np.degrees(nu[5, :]), 'r-', linewidth=2.5, label='Yaw Rate $r$ [deg/s] (Angular Decay)')
    plt.plot(t, nu[1, :], 'b--', linewidth=2, label='Sway Slippage $v$ [m/s] (Cross-coupled Drag)')
    
    plt.title("Section 6.6 Time Realization: 6-DOF Hydrodynamic Momentum Decay", fontsize=11)
    plt.xlabel("Simulation Runtime (Seconds)")
    plt.ylabel("Velocity Magnitudes")
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend()
    plt.show()

widgets.interact(simulate_section_6_6_unforced_system,
                 added_mass_multiplier=widgets.FloatSlider(min=0.2, max=2.0, step=0.2, value=1.0, description='Added Mass %:'),
                 init_yaw_kick_deg=widgets.FloatSlider(min=1.0, max=10.0, step=1.0, value=4.0, description='Init Yaw Kick:'));

interactive(children=(FloatSlider(value=1.0, description='Added Mass %:', max=2.0, min=0.2, step=0.2), FloatSl…

---

## Section 6.7: Maneuvering Properties & Stability Analysis

Once the monolithic matrices are built, Section 6.7 provides the analytical tests used to check if a hull design is inherently **directionally stable** or dangerously unstable in a straight line. 

### Decoupling the Disturbance: Sway ($v$) vs. Yaw Rate ($r$)
When an unsteered submarine is cruising in a straight line, its body-fixed velocities should be purely longitudinal ($u > 0$, while sway $v = 0$ and yaw rate $r = 0$). If a cross-current or an accidental rudder twitch kicks the boat off course, it introduces two distinct hydrodynamic errors:
1. **Yaw Rate ($r$):** The angular velocity telling you how fast the bow is physically swinging left or right (measured in degrees per second).
2. **Sway Velocity ($v$):** The lateral, broadside slippage of the hull through the water column (crabbing sideways).

### The Hydrodynamic Tug-of-War
As soon as these errors appear, a brutal physics battle begins inside the state equations:
* **The Instigator (The Munk Moment):** Because the broadside added mass ($Y_{\dot{v}}$) is massive compared to surge added mass ($X_{\dot{u}}$), the water flowing around the angled hull creates a powerful turning moment. This torque acts as a *negative spring*, pushing in the **same direction** as the swing. It attempts to worsen the turn, forcing the submarine to broadside the fluid.
* **The Restorer (Viscous Cross-Flow Damping):** As the hull slips sideways ($v$) and rotates ($r$), the viscous drag of the water pushes back in the **opposite direction** of the motion. 

If your damping coefficients in $D(\nu)$ are large enough, they overpower the Munk Moment, forcing both $v$ and $r$ back to zero (the vehicle runs straight). If you drop the damping parameters too low, the Munk Moment wins. The angular error accelerates, the sway slippage increases exponentially, and the submarine breaks out into an unforced, permanent circular spinout loop.

Let's simulate this stability threshold interactively.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets  # FIX: Explicit import to eliminate the NameError

def simulate_section_6_7_stability(damping_multiplier):
    """
    Simulates a 3-DOF horizontal trajectory to demonstrate the analytical stability 
    principles of Section 6.7. Shows how low cross-flow drag triggers a spinout.
    """
    t = np.linspace(0, 40, 400)
    dt = t[1] - t[0]
    
    # Structural baselines (1500 ton, 65m submarine configuration)
    m = 1500000.0
    length = 65.0
    iz = (1.0 / 12.0) * m * (length**2)
    
    # Hydrodynamic Added Mass Invariants (Section 6.3)
    X_udot, Y_vdot, N_rdot = m*0.1, m*0.8, iz*0.7
    M = np.diag([m + X_udot, m + Y_vdot, iz + N_rdot])
    M_inv = np.linalg.inv(M)
    
    # Viscous Damping Matrices (Section 6.4) scaled by the interactive slider
    D_l = np.diag([5e3, 3e4, 1.5e5]) * damping_multiplier
    D_n_base = np.diag([2e3, 5e4, 4e5]) * damping_multiplier
    
    # State tracking vectors: eta = [x, y, psi] (Earth coordinates), nu = [u, v, r] (Body speeds)
    eta = np.zeros((3, len(t)))
    nu = np.zeros((3, len(t)))
    
    # INITIAL DISTURBANCE CONDITIONS: 
    # The submarine is cruising at 4 m/s (approx 8 knots) forward (u), 
    # but receives a sudden 2 deg/s yaw rate (r) bump from an external force.
    nu[:, 0] = [4.0, 0.0, np.radians(2.0)]
    
    # Constant forward surge thrust applied (No active steering commands)
    tau = np.array([45000.0, 0.0, 0.0])
    
    for i in range(1, len(t)):
        u, v, r = nu[:, i-1]
        psi = eta[2, i-1]
        
        # Section 6.3 Hydrodynamic Coriolis Cross-coupling (The Destabilizing Munk Moment)
        # Notice how forward speed (u) and sway velocity (v) feed directly into the Yaw torque!
        C_nu = np.array([
            [0.0, 0.0, -M[1, 1] * v],
            [0.0, 0.0,  M[0, 0] * u],
            [M[1, 1] * v, -M[0, 0] * u, 0.0]  # <-- The matrix index driving the spinout torque
        ])
        
        # Blend linear and absolute velocity quadratic damping
        D_total = D_l + np.diag(np.abs(nu[:, i-1])) @ D_n_base
        
        # Acceleration step: nu_dot = M^-1 * (Thrust - Coriolis_Torque - Damping_Torque)
        nu_dot = M_inv @ (tau - C_nu @ nu[:, i-1] - D_total @ nu[:, i-1])
        nu[:, i] = nu[:, i-1] + nu_dot * dt
        
        # Kinematic transformation to Earth frame (NED) to draw the ground track
        R = np.array([
            [np.cos(psi), -np.sin(psi), 0.0],
            [np.sin(psi),  np.cos(psi), 0.0],
            [0.0,         0.0,          1.0]
        ])
        eta[:, i] = eta[:, i-1] + (R @ nu[:, i]) * dt

    # Plot Layout
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
    
    # Plot 1: XY Position Trace (Ground Track)
    ax1.plot(eta[1, :], eta[0, :], color='darkblue', linewidth=2.5, label='Submarine Path')
    ax1.scatter(eta[1, 0], eta[0, 0], color='green', s=50, label='Disturbance Injection Point', zorder=5)
    ax1.set_title("Section 6.7 Horizontal Maneuvering Track (North vs East)", fontsize=11)
    ax1.set_xlabel("East Position $y$ [Meters]")
    ax1.set_ylabel("North Position $x$ [Meters]")
    ax1.axis('equal')
    ax1.grid(True, linestyle=':')
    ax1.legend()
    
    # Plot 2: Angular State Convergence vs. Divergence
    # Shows the exact historical battle between Yaw Rate (r) and Sway Slippage (v)
    ax2.plot(t, np.degrees(nu[2, :]), color='crimson', linewidth=2.5, label='Yaw Rate $r$ [deg/s] (Rotation)')
    ax2.plot(t, nu[1, :], color='dodgerblue', linestyle='--', linewidth=2, label='Sway Velocity $v$ [m/s] (Sideways Slip)')
    ax2.axhline(0, color='black', linestyle='-', alpha=0.3)
    ax2.set_title("Evolution of Sway Slip vs. Angular Turning Rate", fontsize=11)
    ax2.set_xlabel("Time Elapsed [Seconds]")
    ax2.set_ylabel("State Amplitude")
    ax2.grid(True, linestyle=':')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

# Run the interactive widget framework cleanly
widgets.interact(simulate_section_6_7_stability,
                 damping_multiplier=widgets.FloatSlider(min=0.1, max=2.0, step=0.1, value=1.0, description='Damp Matrix %:'));

interactive(children=(FloatSlider(value=1.0, description='Damp Matrix %:', max=2.0, min=0.1), Output()), _dom_…

---

## Chapter 6 Summary: The Operational Plant Blueprint

With Section 6.7 fully simulated, we have completed the baseline unforced hydrodynamic model of our marine craft. Here is how Fossen's sections fit together logically inside a GNC architecture:

1. **Section 6.2 ($M_{RB}$)** sets the raw structural inertia of your dry hull plating and internal decks.
2. **Section 6.3 ($A, C_A$)** layer on the invisible mass of the water column. This introduces the **Munk Moment**, proving mathematically that an unsteered slender submarine is inherently directionally unstable when drifting.
3. **Section 6.4 ($D$)** provides the exact physical counterweight to that instability, blending linear skin friction with quadratic cross-flow drag to bound the vehicle's angular state transitions.
4. **Section 6.5 ($g$)** acts as the pendulum anchor, utilizing metacentric height ($z_g$) to preserve stability in Roll and Pitch.
5. **Section 6.6** compiles these distinct matrices into a unified, modular plant loop ready to receive control vectors.

We have successfully mapped out how the vehicle behaves on its own in a calm fluid. Tomorrow morning, we expand this exact matrix architecture by defining $\tau$: injecting active control plane lift profiles and environmental force models.